# Lab Exploration 3 - G-Eval

For this lab, you will experiment with G-Eval. G-Eval is an LLM-as-a-judge framework that uses chain-of-thought prompting under the hood to evaluate on any custom criteria. You should be familiar with the respective paper we are reading in class: [G-EVAL: NLG Evaluation using GPT-4 with Better Human Alignment](https://arxiv.org/pdf/2303.16634)



G-Eval is available through DeepEval, an evaluation platform. You should start by spending some time reading through the G-Eval docs to get an understanding of how the library works: [G-Eval](https://deepeval.com/docs/metrics-llm-evals).

We are expecting you to spend ~5 hours on this assignment exploring and trying to use G-Eval.

For this lab, your tasks are to:
- Read through the G-Eval documentation: https://deepeval.com/docs/metrics-llm-evals
- Get the sample code below working
- Pick at least 3 metrics (correctness, coherence, tonality, creativity, etc., whatever you can think of) that you think would be interesting to experiment with.
- Pick at least 1 model to experiment with. The example below uses Qwen.
- Setup your metrics and experiment with them. You should explore the following questions:
  1. What is the difference between providing criteria and evaluation_steps? Do you notice performance differences?
  2. How does providing a rubric affect evaluations?
  3. How well does the model perform on each metric? Do you agree with the model's assessments? Would you trust this LLM-as-a-judge approach at scale?
  4. If you are dissatisfied with the performance of the judge, what can you do to make it better? Is there something you need to change in the criteria/evaluation_steps, test_case, or something else?

Some other options for exploration:
- Compare several different LLM judges. You can find many different kinds of models on HuggingFace. How do they differ in performance and judgement?
- Find or create a dataset of 5-10 samples for a given metric, run G-Eval across the data, and also complete the task yourself. What is your inter-rater agreement with the LLM?
- [GEval has an option to override the default templates](https://deepeval.com/docs/metrics-llm-evals#customize-your-template). This may be interesting to experiment with.

Below is some sample code for getting starting using G-Eval:

In [1]:
!pip install deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.1/824.1 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 4.2 MB/s eta 0:00:00


In [2]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from deepeval.models.base_model import DeepEvalBaseLLM

By default DeepEval looks for an OpenAI API key to use as the backend LLM in G-Eval. However, you can also load HuggingFace models:

In [3]:
class HFModel(DeepEvalBaseLLM):
    def __init__(self, model_name):
        self.device = "cuda" # the device to load the model onto
        self.model = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model_name = model_name

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        model = self.load_model()

        messages = [
            {"role": "user", "content": prompt}
        ]

        model_inputs = self.tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True
        ).to(self.device)

        generated_ids = model.generate(input_ids=model_inputs['input_ids'], attention_mask=model_inputs['attention_mask'], max_new_tokens=300, do_sample=True)
        # Only decode the new tokens, not the input prompt
        response_ids = generated_ids[:, model_inputs['input_ids'].shape[1]:]
        return self.tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0]

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return self.model_name

Here are a few models you can start out experimenting with. You should be able to load all of these on a 40GB A100. I was also able to run all of these on a 15GB T4, but Qwen is pretty tight.

Heads up: you will likely find that TinyLlama does not work well with G-Eval, because it's not very good at returning properly formatted JSON. [There is apparently a way that you can improve this](https://deepeval.com/guides/guides-using-custom-llms#json-confinement-libraries), but I couldn't get it to work. If you don't have enough resources to run Qwen, you can 1). Try to find a smaller model in the 1B-3B parameter range that works, 2). try to use a JSON confinement library, or 3). ditch G-Eval and experiment with LLM-as-a-judge through pure prompting. This is also a valid approach for this assignment.

In [4]:
tinyLlama = HFModel("TinyLlama/TinyLlama-1.1B-Chat-v1.0") # 1.1B parameters

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [5]:
qwen = HFModel("Qwen/Qwen2.5-7B-Instruct") # 7B parameters

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
# We can just test calling the models directly
testText = "Evaluate the creativity of this sentence on a scale of 1-10: 'Now the other princes of the Achaeans slept soundly the whole night through, but Agamemnon son of Atreus was troubled, so that he could get no rest.'"

print("TinyLlama:")
print(tinyLlama.generate(testText))

print("Qwen:")
print(qwen.generate(testText))

TinyLlama:
I do not have the capability to rate/score creativity. However, according to the given passage, the creativity of the sentence in question can be evaluated on a scale ranging from 1-10. As the sentence is a basic grammatical structure, it follows the rule of having subject and predicate. In this case, 'now the other princes of the Achaeans slept soundly the whole night through' effectively has a logical progression as it is related to the subject (princes) completing a task (sleeping soundly). This structure also provides enough detail for readers to understand what was going through Agamemnon's mind, as he is experiencing an intermittent sleep deprivation. Overall, the sentence falls in the 7-9 range on the creativity scale for a simple and straightforward sentence.
Qwen:
To evaluate the creativity of the sentence, let's break it down and consider its elements:

1. **Imagery**: The sentence paints a vivid picture by contrasting the peaceful sleep of the other princes with A

Now, let's try GEval.

Notice that GEval can take either a `criteria` parameter or `evaluation_steps`. If it's only given `criteria`, it will generate it's own `evaluation_steps`, like what is done in the original paper.

In [8]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

name = "Accuracy"
criteria = "Determine whether the actual output is factually accurate and correct based on the expected output. Vague language or syntactic difference are OK. Ignore sentence structure differences."

correctness_metric = GEval(
    name=name,
    criteria=criteria,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    model=qwen,
    verbose_mode=True
)

In [9]:
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input="The dog chased the cat up the tree, who ran up the tree?",
    actual_output="It would be the cat.",
    expected_output="The cat."
)

In [10]:
correctness_metric.measure(test_case)

Output()

**************************************************

Accuracy [GEval] Verbose Logs

**************************************************

Criteria:
Determine whether the actual output is factually accurate and correct based on the expected output. Vague language 
or syntactic difference are OK. Ignore sentence structure differences. 
 
Evaluation Steps:
[
    "Check if the actual output contains the same factual information as the expected output, ignoring sentence 
structure differences.",
    "Verify that any statements in the actual output match those in the expected output without considering 
syntactic variations.",
    "Ensure there are no contradictions between the actual output and the expected output."
] 
 
Rubric:
None 
 
Score: 0.0
 
Reason: The actual output 'It would be the cat.' does not match the expected output 'The cat.', leading to a 
contradiction. The response fails to align with the expected factual information.

======================================================================

0.0

In [11]:
# Let's try a different metric
name = "Creativity"
criteria = "Determine how creative the given sentence is, on a scale of 1-10."

creativity_metric = GEval(
    name=name,
    criteria=criteria,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=qwen,
    verbose_mode=True
)

In [12]:
test_case_2 = LLMTestCase(
    input="", # Add an empty string for the required input parameter
    actual_output="Now the other princes of the Achaeans slept soundly the whole night through, but Agamemnon son of Atreus was troubled, so that he could get no rest."
)

In [13]:
creativity_metric.measure(test_case_2)

Output()

**************************************************

Creativity [GEval] Verbose Logs

**************************************************

Criteria:
Determine how creative the given sentence is, on a scale of 1-10. 
 
Evaluation Steps:
[
    "Identify unique and novel elements in the sentence.",
    "Assess the originality of the sentence's structure.",
    "Evaluate the use of metaphors, similes, or other literary devices.",
    "Rate the overall creativity on a scale of 1-10 based on the above assessments."
] 
 
Rubric:
None 
 
Score: 0.4
 
Reason: The response lacks unique and novel elements, showing a typical narrative structure without significant 
originality. It does not demonstrate the use of metaphors, similes, or other literary devices, which results in a 
lower score.

======================================================================

0.4

In [20]:
# Experimentation here
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

frenchness_metric = GEval(
    name="Frenchness",
    # criteria="Determine how French the output is based on its Frenchiness.",
    evaluation_steps=[
        "Pontificate upon what it means for something to be 'French,' both physically and spiritually.",
        "List criteria by which Frenchiness may be measured.",
        "Score the Frenchness of the output based on the criteria you listed, on a scale from 1 to 10."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=qwen,
    verbose_mode=True
)

In [21]:
frenchness_metric.measure(test_case_2)

Output()

**************************************************

Frenchness [GEval] Verbose Logs

**************************************************

Criteria:
None 
 
Evaluation Steps:
[
    "Pontificate upon what it means for something to be 'French,' both physically and spiritually.",
    "List criteria by which Frenchiness may be measured.",
    "Score the Frenchness of the output based on the criteria you listed, on a scale from 1 to 10."
] 
 
Rubric:
None 
 
Score: 0.0
 
Reason: The response does not address the concept of 'Frenchness' or provide any criteria for measuring it. It 
instead references ancient Greek literature, specifically the Iliad, which is unrelated to the evaluation steps.

======================================================================

0.0

In [22]:
test_case_3 = LLMTestCase(
    input="", # Add an empty string for the required input parameter
    actual_output="Oui oui, baguettes and escargots."
)

In [23]:
frenchness_metric.measure(test_case_3)

Output()

**************************************************

Frenchness [GEval] Verbose Logs

**************************************************

Criteria:
None 
 
Evaluation Steps:
[
    "Pontificate upon what it means for something to be 'French,' both physically and spiritually.",
    "List criteria by which Frenchiness may be measured.",
    "Score the Frenchness of the output based on the criteria you listed, on a scale from 1 to 10."
] 
 
Rubric:
None 
 
Score: 0.1
 
Reason: The response is brief and touches on stereotypical French foods but fails to address the physical and 
spiritual aspects of Frenchness as required by step 1. It also does not list any criteria for measuring Frenchness,
missing step 2 entirely. As such, it scores low on meeting the evaluation criteria.

======================================================================

0.1